In [21]:
# imports
from langchain_community.document_loaders import WebBaseLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_ollama.embeddings import OllamaEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_core.prompts import ChatPromptTemplate
from langchain_ollama.chat_models import ChatOllama
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser
from langchain_classic.chains import combine_documents, create_retrieval_chain


In [3]:
# We will load content from Wikipedia page on LLM
loader = WebBaseLoader("https://en.wikipedia.org/wiki/Large_language_model")
document = loader.load()

# split text into smaller chunks
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000, chunk_overlap=200, length_function=len
)
documents = text_splitter.split_documents(document)

# embedding
embeddings = OllamaEmbeddings(model="nomic-embed-text")

# using FAISS (Facebook AI Similarity Search) for vectors
# use either faiss-gpu or faiss-cpu
vector_store = FAISS.from_documents(documents, embeddings)

In [4]:
# similarity search
result = vector_store.similarity_search("What are the challenges faced by large language models?")
print(result[0].page_content)

A large language model (LLM) is a neural network trained on a vast amount of text for natural language processing tasks, especially language generation. LLMs can generate, summarize, translate and parse text in many contexts, and are a foundational technology behind modern chatbots.[1] Biased or inaccurate training data can make an LLM's output less reliable.[2]
As of 2024, the largest and most capable LLMs are all based on transformer architectures,[3] which, according to the 2017 paper Attention Is All You Need, can be more efficient and parallelizable than earlier statistical and recurrent neural network models.[4] Research into other architectures, such as state space models, is ongoing.[5]
Benchmark evaluations for LLMs attempt to measure model reasoning, factual accuracy, alignment, and safety.[6]


In [6]:
# get retriever
retriever = vector_store.as_retriever()

# llm
llm = ChatOllama(model="gemma4:e4b")

In [ ]:
# METHOD 1
# document parsing function to string
def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)


# create prompt template
prompt_template = ChatPromptTemplate.from_template(
    """
    You are expert assistant. Answer the question based only the provided context, and if you can't find then simply say I don't have the answer.
    
    Context: {context}

    Question: {query}
    """
)

# chain
rag_chain = (
    {"context": retriever | format_docs, "query": RunnablePassthrough()}
    | prompt_template
    | llm
    | StrOutputParser()
)

In [20]:
rag_chain.invoke("One specific challenge faced by large language models?")

'Specific challenges faced by large language models include:\n\n*   Their output can be less reliable if the training data is biased or inaccurate.\n*   There are concerns that frequent use of large language models could weaken critical thinking.'

In [19]:
# METHOD 2

# create prompt template
prompt_template = ChatPromptTemplate.from_template(
    """
    You are expert assistant. Answer the question based only the provided context, and if you can't find then simply say I don't have the answer.
    
    <context>
    {context}
    </context>

    Question: {input}
    """
)

# create document chain
document_chain = combine_documents.create_stuff_documents_chain(
    llm=llm, prompt=prompt_template
)

# retrieval chain
retreival_chain = create_retrieval_chain(retriever, document_chain)

result = retreival_chain.invoke(
    {"input": "One specific challenge faced by large language models?"}
)

print(f"Answer: {result["answer"]}")

Answer: An LLM's output may be less reliable if the training data is biased or inaccurate.
